# Apache Iceberg Lakehouse — Student Notebook

1. **Setup** — configuration, S3 Tables catalog, upload scripts and data
2. **Bronze** — run Glue jobs, validate with Athena
3. **Silver** — run Glue job, validate with Athena
4. **Gold** — run Glue job, validate with Athena
5. **Iceberg Features** — snapshots, time-travel

---

# Part 1: Setup

## Step 1: Configuration

Set up all resource names matching the CloudFormation stack outputs. Bronze uses a
standard S3 bucket with Glue Catalog Iceberg tables. Silver and Gold use **S3 Tables**
(managed Iceberg) accessed through the `s3tablescatalog` federated catalog.

Fill in the values from your CloudFormation stack outputs.

In [ ]:
import os, time
import boto3
import pandas as pd

AWS_REGION = 'us-east-1'

# TODO: Fill in from CloudFormation stack outputs
BUCKET = 'ecommerce-analytics-datalake-dev-316200272570'              # DataLakeBucketName output
DATABASE = 'ecommerce_analytics_bronze_dev'
ATHENA_WORKGROUP = 'ecommerce-analytics-workgroup-dev'      # AthenaWorkgroupName output

# S3 Tables catalog (silver + gold layers)
TABLE_BUCKET_NAME = 'ecommerce-analytics-gold-dev-316200272570'  # from GoldTableBucketARN output (bucket name part)
S3T_CATALOG = f's3tablescatalog/{TABLE_BUCKET_NAME}'
SILVER_DB = 'ecommerce_silver'
GOLD_DB = 'ecommerce_gold'

# Glue job names
CDC_JOB = 'ecommerce-analytics-cdc-bronze-dev'
EVENTS_JOB = 'ecommerce-analytics-events-bronze-dev'
SILVER_JOB = 'ecommerce-analytics-silver-layer-dev'
GOLD_JOB = 'ecommerce-analytics-gold-layer-dev'

s3 = boto3.client('s3', region_name=AWS_REGION)
glue = boto3.client('glue', region_name=AWS_REGION)
athena = boto3.client('athena', region_name=AWS_REGION)

print('\u2705 Configuration set')

## Step 2: Register S3 Tables Catalog (one-time after stack deploy)

The Silver and Gold layers write to **S3 Tables** (managed Iceberg). To query these
tables from Athena and allow Glue jobs to discover them, we must register a
`s3tablescatalog` federated catalog in the Glue Data Catalog. This bridges the
S3 Tables API with AWS analytics services (Athena, Glue, EMR). The catalog uses
`IAM_ALLOWED_PRINCIPALS` so access is controlled by IAM policies, not Lake Formation grants.

This only needs to run once per account/region — it's safe to re-run (idempotent).

In [ ]:
ACCOUNT_ID = boto3.client('sts', region_name=AWS_REGION).get_caller_identity()['Account']

try:
    glue.create_catalog(
        Name='s3tablescatalog',
        CatalogInput={
            'FederatedCatalog': {
                'Identifier': f'arn:aws:s3tables:{AWS_REGION}:{ACCOUNT_ID}:bucket/*',
                'ConnectionName': 'aws:s3tables'
            },
            'CreateDatabaseDefaultPermissions': [
                {'Principal': {'DataLakePrincipalIdentifier': 'IAM_ALLOWED_PRINCIPALS'}, 'Permissions': ['ALL']}
            ],
            'CreateTableDefaultPermissions': [
                {'Principal': {'DataLakePrincipalIdentifier': 'IAM_ALLOWED_PRINCIPALS'}, 'Permissions': ['ALL']}
            ],
            'AllowFullTableExternalDataAccess': 'True'
        }
    )
    print('✅ s3tablescatalog registered')
except glue.exceptions.AlreadyExistsException:
    print('✅ s3tablescatalog already exists')

## Step 3: Upload Glue Scripts and Datasets

Upload the four Glue ETL scripts and the CDC/events datasets to S3. The Glue jobs
fetch their scripts from `s3://{bucket}/glue-scripts/` at runtime, so this must
happen before running any jobs.

In [ ]:
import glob as globmod

# Upload Glue scripts
scripts = {
    'glue-jobs/cdc-batch/src/iceberg-cdc-bronze-layer.py': 'glue-scripts/iceberg-cdc-bronze-layer.py',
    'glue-jobs/events/src/iceberg-events-bronze-layer.py': 'glue-scripts/iceberg-events-bronze-layer.py',
    'glue-jobs/silver/src/iceberg-silver-layer.py': 'glue-scripts/iceberg-silver-layer.py',
    'glue-jobs/gold/src/iceberg-gold-layer.py': 'glue-scripts/iceberg-gold-layer.py',
}
for local, s3_key in scripts.items():
    s3.upload_file(local, BUCKET, s3_key)
    print(f'\u2705 Uploaded {s3_key}')

# Upload datasets
for local_path in globmod.glob('datasets/**/*', recursive=True):
    if os.path.isfile(local_path):
        s3.upload_file(local_path, BUCKET, local_path)

resp = s3.list_objects_v2(Bucket=BUCKET, Prefix='datasets/', MaxKeys=5)
for obj in resp.get('Contents', []):
    print(f'  {obj["Key"]} ({obj["Size"]:,} bytes)')
print('\u2705 Datasets uploaded')

## Step 4: Helper Functions

`run_glue_job` starts a Glue job and polls until completion. `run_query` executes
Athena SQL and returns a DataFrame. The optional `catalog` and `database` parameters
let us query S3 Tables (silver/gold) by passing the federated catalog name.

In [ ]:
def run_glue_job(job_name):
    run = glue.start_job_run(JobName=job_name)
    run_id = run['JobRunId']
    print(f'\u25b6 Started {job_name} (run: {run_id})')
    while True:
        status = glue.get_job_run(JobName=job_name, RunId=run_id)['JobRun']
        state = status['JobRunState']
        if state in ('SUCCEEDED', 'FAILED', 'STOPPED', 'ERROR', 'TIMEOUT'):
            break
        print(f'  {state}...', end='\r')
        time.sleep(30)
    if state == 'SUCCEEDED':
        print(f'\u2705 {job_name} SUCCEEDED ({status.get("ExecutionTime", 0)}s)')
    else:
        print(f'\u274c {job_name} {state}: {status.get("ErrorMessage", "")}')
    return state

def run_query(sql, catalog=None, database=None):
    params = {
        'QueryString': sql,
        'WorkGroup': ATHENA_WORKGROUP,
        'ResultConfiguration': {'OutputLocation': f's3://{BUCKET}/athena-results/'}
    }
    if catalog or database:
        params['QueryExecutionContext'] = {}
        if catalog:
            params['QueryExecutionContext']['Catalog'] = catalog
        if database:
            params['QueryExecutionContext']['Database'] = database
    exe = athena.start_query_execution(**params)
    qid = exe['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(1)
    if state != 'SUCCEEDED':
        raise Exception(status['QueryExecution']['Status'].get('StateChangeReason', state))
    results = athena.get_query_results(QueryExecutionId=qid)
    cols = [c['Label'] for c in results['ResultSet']['ResultSetMetadata']['ColumnInfo']]
    rows = [[f['VarCharValue'] if 'VarCharValue' in f else None for f in r['Data']] for r in results['ResultSet']['Rows'][1:]]
    return pd.DataFrame(rows, columns=cols)

print('\u2705 Helpers ready')

---
# Part 2: Bronze Layer

## Step 5: Run Bronze Glue Jobs

Two jobs ingest raw data into Iceberg tables on S3:
- **CDC Bronze** — reads Debezium CDC JSON, deduplicates by primary key, and uses
  `MERGE INTO` to upsert orders, users, and products.
- **Events Bronze** — reads clickstream JSON and appends to the events table.

In [ ]:
run_glue_job(CDC_JOB)
run_glue_job(EVENTS_JOB)

## Step 6: Validate Bronze Layer

Write Athena queries to validate the bronze tables:
1. Count rows in each table (orders, users, products, events)
2. Check for duplicate order IDs (should be 0 after MERGE dedup)
3. Show event type distribution

Use `run_query()` — bronze tables are in the default Glue catalog, so no catalog parameter needed.
Reference tables as `{DATABASE}.table_name`.

In [ ]:
bronze_counts = run_query(f"""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM {DATABASE}.orders
    UNION ALL
    SELECT 'users', COUNT(*) FROM {DATABASE}.users
    UNION ALL
    SELECT 'products', COUNT(*) FROM {DATABASE}.products
    UNION ALL
    SELECT 'events', COUNT(*) FROM {DATABASE}.events
""")
print(bronze_counts.to_string(index=False))

In [ ]:
dupes = run_query(f"SELECT order_id, COUNT(*) AS c FROM {DATABASE}.orders GROUP BY order_id HAVING COUNT(*) > 1")
print(f'Duplicate orders: {len(dupes)}')

In [ ]:
event_types = run_query(f"SELECT event_type, COUNT(*) AS cnt FROM {DATABASE}.events GROUP BY event_type ORDER BY cnt DESC")
print(event_types.to_string(index=False))

---
# Part 3: Silver Layer

## Step 7: Run Silver Glue Job

Reads from Bronze (Glue Catalog / S3) and writes enriched tables to **S3 Tables**:
- **order_details** — orders joined with users, adds customer segments and lifetime value
- **enriched_events** — events joined with users and products, adds revenue and session sequence
- **product_performance** — aggregated metrics per product (views, carts, purchases, conversion rates)

In [ ]:
run_glue_job(SILVER_JOB)

## Step 8: Validate Silver Layer

Write Athena queries to validate the silver tables:
1. Count rows in each table (order_details, enriched_events, product_performance)
2. Show customer segment distribution from order_details
3. Show product funnel conversion rates from product_performance

Silver tables are on **S3 Tables** — pass `catalog=S3T_CATALOG, database=SILVER_DB`
to `run_query()`. Use unqualified table names (no database prefix in SQL).

In [ ]:
silver_counts = run_query("""
    SELECT 'order_details' AS table_name, COUNT(*) AS row_count FROM order_details
    UNION ALL
    SELECT 'enriched_events', COUNT(*) FROM enriched_events
    UNION ALL
    SELECT 'product_performance', COUNT(*) FROM product_performance
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(silver_counts.to_string(index=False))

In [ ]:
segments = run_query("""
    SELECT
        customer_segment,
        COUNT(*) AS order_count,
        ROUND(AVG(total_amount), 2) AS avg_order_value
    FROM order_details
    GROUP BY customer_segment
    ORDER BY order_count DESC
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(segments.to_string(index=False))

In [ ]:
funnel = run_query("""
    SELECT
        category,
        SUM(page_views) AS page_views,
        SUM(purchases) AS purchases,
        ROUND(CAST(SUM(purchases) AS DOUBLE) / NULLIF(SUM(page_views), 0), 4) AS conversion_rate
    FROM product_performance
    GROUP BY category
    ORDER BY page_views DESC
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(funnel.to_string(index=False))

---
# Part 4: Gold Layer

## Step 9: Run Gold Glue Job

Reads from Silver (S3 Tables) and writes business-ready aggregations to **S3 Tables**:
- **customer_analytics** — monthly customer metrics (revenue, segments, geography)
- **realtime_metrics** — hourly event metrics (traffic sources, conversions, revenue)

In [ ]:
run_glue_job(GOLD_JOB)

## Step 10: Validate Gold Layer

Write Athena queries to validate the gold tables:
1. Show monthly customer analytics (analysis_month, total_customers, total_revenue, avg_order_value)
2. Show recent hourly metrics (metric_hour, total_events, unique_users, hourly_revenue)

Gold tables are on **S3 Tables** — pass `catalog=S3T_CATALOG, database=GOLD_DB`.

In [ ]:
customer_analytics = run_query("""
    SELECT
        analysis_month,
        total_customers,
        total_revenue,
        avg_order_value
    FROM customer_analytics
    ORDER BY analysis_month
""", catalog=S3T_CATALOG, database=GOLD_DB)
print(customer_analytics.to_string(index=False))

In [ ]:
realtime_metrics = run_query("""
    SELECT
        metric_hour,
        total_events,
        unique_users,
        hourly_revenue
    FROM realtime_metrics
    ORDER BY metric_hour DESC
    LIMIT 10
""", catalog=S3T_CATALOG, database=GOLD_DB)
print(realtime_metrics.to_string(index=False))

---
# Part 5: Iceberg Features

## Step 11: Snapshot Inspection

Iceberg maintains a log of every change as immutable **snapshots**. Query the
`$snapshots` and `$history` metadata tables to see when data was written,
what operation occurred (append, overwrite), and how many files were added.

Use the bronze orders table for this — reference metadata tables as:
- `"{DATABASE}"."orders$snapshots"`
- `"{DATABASE}"."orders$history"`

In [ ]:
snapshots = run_query(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM "{DATABASE}"."orders$snapshots"
    ORDER BY committed_at
""")
print(snapshots.to_string(index=False))

In [ ]:
history = run_query(f"""
    SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor
    FROM "{DATABASE}"."orders$history"
    ORDER BY made_current_at
""")
print(history.to_string(index=False))

## Step 12: Time-Travel Query

Iceberg's time-travel lets you query data **as it existed at a previous point in time**
using `FOR TIMESTAMP AS OF`. To demonstrate this, we need data that **changes between
snapshots**. Since each job only ran once, all snapshots contain the same data.

The approach:
1. Query the current events row count
2. Get the latest snapshot timestamp from the `events$snapshots` metadata table
3. **Re-run the Events Bronze job** to append a second batch (INSERT INTO creates a new snapshot with ~2x rows)
4. Query the current state again (should show doubled rows)
5. Use `FOR TIMESTAMP AS OF` with the snapshot timestamp to read the pre-reload state

In [ ]:
# 1. Current events row count
count_before = run_query(f"SELECT COUNT(*) AS rows FROM {DATABASE}.events").iloc[0]['rows']
print(f"Events before reload: {count_before}")

# 2. Latest snapshot timestamp (the pre-reload state)
snap_ts = run_query(f'SELECT MAX(committed_at) AS ts FROM "{DATABASE}"."events$snapshots"').iloc[0]['ts']
print(f"Latest snapshot before reload: {snap_ts}")

# 3. Re-run the events job to append a second batch
run_glue_job(EVENTS_JOB)

# 4. Current state again (should show ~2x rows)
count_current = run_query(f"SELECT COUNT(*) AS rows FROM {DATABASE}.events").iloc[0]['rows']
print(f"Events after reload: {count_current}")

# 5. Time-travel query: read the pre-reload state
count_historical = run_query(f"""SELECT COUNT(*) AS rows FROM {DATABASE}.events
    FOR TIMESTAMP AS OF TIMESTAMP '{snap_ts}'""").iloc[0]['rows']

# Current vs historical, side by side
comparison = pd.DataFrame({
    'state': ['current', f'historical (as of {snap_ts})'],
    'row_count': [count_current, count_historical]
})
print(comparison.to_string(index=False))


---
## Step 13: Medallion Architecture Summary

Write a summary cell that queries all three layers and prints row counts.
Bronze uses the default catalog; Silver and Gold use `S3T_CATALOG`.

In [ ]:
layers = [
    ('Bronze', ['orders', 'users', 'products', 'events'], {}),
    ('Silver', ['order_details', 'enriched_events', 'product_performance'],
     {'catalog': S3T_CATALOG, 'database': SILVER_DB}),
    ('Gold', ['customer_analytics', 'realtime_metrics'],
     {'catalog': S3T_CATALOG, 'database': GOLD_DB}),
]

for layer, tables, kwargs in layers:
    print(f'\n{layer}')
    for table in tables:
        # Bronze tables need the database prefix; silver and gold use the query context
        name = f'{DATABASE}.{table}' if layer == 'Bronze' else table
        count = run_query(f'SELECT COUNT(*) AS cnt FROM {name}', **kwargs).iloc[0]['cnt']
        print(f'  {table:<22} {int(count):>10,} rows')